In [1]:
import os
import bs4
from dotenv import load_dotenv

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\amroosha\AppData\Local\Temp\ipykernel_23568\4135347378.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [2]:
tracer = LangChainTracer()

# Multi-representation Indexing

In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [5]:
print(len(docs))
print(docs[0].page_content[:500])


2






LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: 


In [8]:
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatGroq(model="openai/gpt-oss-20b", temperature=0)
    | StrOutputParser()
)

for doc in docs:
    doc.page_content = doc.page_content[:4000]

summaries = [chain.invoke(doc, config={"callbacks": [tracer]}) for doc in docs]

print(summaries)

['**LLM‑Powered Autonomous Agents – Key Takeaways**\n\n| Section | Core Idea |\n|---------|-----------|\n| **Agent System Overview** | An autonomous agent uses a large language model (LLM) as its “brain” and relies on three main modules: planning, memory, and tool use. |\n| **Planning** | • **Task decomposition** – the LLM breaks a complex goal into smaller sub‑goals (e.g., via Chain‑of‑Thought or Tree‑of‑Thought prompting). <br>• **Self‑reflection** – the agent critiques its own actions, learns from mistakes, and refines future steps. |\n| **Memory** | • **Short‑term** – in‑context learning (prompting) acts as a temporary working memory. <br>• **Long‑term** – an external vector store with fast retrieval (e.g., MIPS) lets the agent retain and recall vast amounts of information over time. |\n| **Tool Use** | The agent calls external APIs or tools to supplement knowledge that isn’t in the LLM’s weights (current data, code execution, proprietary sources, etc.). |\n| **Planning Techniques*